# 📊 Telecom Customer Segmentation & Churn Analysis
### End-to-End ML Pipeline | Tavishi Jain

**Dataset:** `telecom_churn.csv` — 243,553 customers across 4 Indian telecom partners  
**Objective:** Identify customer segments and predict churn using EDA, K-Means clustering, and Random Forest classification.

---
## Pipeline Overview
1. Data Loading & Inspection  
2. Data Cleaning & Feature Engineering  
3. Exploratory Data Analysis (EDA)  
4. Customer Segmentation via K-Means Clustering  
5. Churn Prediction Model (Random Forest)  
6. Class Imbalance Handling (SMOTE)  
7. Model Evaluation: Confusion Matrix, ROC-AUC, Cross-Validation  
8. Feature Importance & Business Insights  


## 1. 📦 Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_curve, roc_auc_score,
                             ConfusionMatrixDisplay)
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
try:
    from imblearn.over_sampling import SMOTE
except ModuleNotFoundError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "imbalanced-learn"])
    from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

# ── Styling ──
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 14, 'axes.labelsize': 12})
CHURN_COLORS = {0: '#2196F3', 1: '#F44336'}  # Blue = No churn, Red = Churn

print("✅ All libraries imported successfully")

ModuleNotFoundError: No module named 'imblearn'

## 2. 📂 Data Loading & Inspection

In [ ]:
df = pd.read_csv('telecom_churn.csv')
print(f"Shape: {df.shape}")
df.head()

In [ ]:
print("=== Data Types ===")
print(df.dtypes)
print(f"\n=== Missing Values ===")
print(df.isnull().sum())

In [ ]:
print("=== Statistical Summary ===")
df.describe().round(2)

In [ ]:
print("=== Churn Distribution ===")
churn_counts = df['churn'].value_counts()
churn_pct = df['churn'].value_counts(normalize=True) * 100
print(pd.DataFrame({'Count': churn_counts, 'Percentage (%)': churn_pct.round(2)}))

## 3. 🛠️ Data Cleaning & Feature Engineering

**Key steps:**
- Fix negative `data_used` values (data anomaly)
- Parse `date_of_registration` → extract `tenure_months`
- Create behavioral feature: `usage_score`
- Encode categoricals


In [ ]:
# Fix anomalous negative data_used values
df['data_used'] = df['data_used'].clip(lower=0)

# Parse registration date and compute tenure
df['date_of_registration'] = pd.to_datetime(df['date_of_registration'])
REFERENCE_DATE = pd.Timestamp('2024-01-01')
df['tenure_months'] = ((REFERENCE_DATE - df['date_of_registration']).dt.days / 30).astype(int)

# Usage score: composite of calls, sms, data (normalized sum)
df['usage_score'] = (
    (df['calls_made'] / df['calls_made'].max()) +
    (df['sms_sent']   / df['sms_sent'].max())   +
    (df['data_used']  / df['data_used'].max())
).round(4)

# Age groups for segmentation
df['age_group'] = pd.cut(df['age'], bins=[18,30,45,60,75],
                         labels=['18-30','31-45','46-60','61-75'])

print("✅ Feature engineering complete")
print(f"  - tenure_months range: {df['tenure_months'].min()} – {df['tenure_months'].max()} months")
print(f"  - Negative data_used clipped: done")
print(f"  - usage_score range: {df['usage_score'].min():.3f} – {df['usage_score'].max():.3f}")
df[['tenure_months','usage_score','age_group']].head()

## 4. 📊 Exploratory Data Analysis (EDA)

### 4.1 Churn Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Pie chart
churn_counts = df['churn'].value_counts()
axes[0].pie(churn_counts, labels=['No Churn','Churned'],
            autopct='%1.1f%%', colors=['#2196F3','#F44336'],
            startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2))
axes[0].set_title('Overall Churn Split', fontweight='bold')

# Bar by telecom partner
churn_by_partner = df.groupby('telecom_partner')['churn'].mean() * 100
churn_by_partner.sort_values().plot(kind='barh', ax=axes[1],
                                     color=['#4CAF50','#FF9800','#2196F3','#9C27B0'])
axes[1].set_xlabel('Churn Rate (%)')
axes[1].set_title('Churn Rate by Telecom Partner', fontweight='bold')
axes[1].xaxis.set_major_formatter(mtick.PercentFormatter())

plt.tight_layout()
plt.savefig('plot_churn_overview.png', bbox_inches='tight')
plt.show()
print("📌 ~20% overall churn rate. Fairly balanced across operators.")

### 4.2 Tenure & Age Distribution by Churn

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Tenure boxplot
sns.boxplot(x='churn', y='tenure_months', data=df, ax=axes[0],
            palette=CHURN_COLORS)
axes[0].set_xticklabels(['No Churn','Churned'])
axes[0].set_title('Tenure Months vs Churn', fontweight='bold')
axes[0].set_xlabel('')

# Age histogram by churn
for val, color in CHURN_COLORS.items():
    subset = df[df['churn'] == val]['age']
    axes[1].hist(subset, bins=20, alpha=0.6, color=color,
                 label='No Churn' if val==0 else 'Churned', density=True)
axes[1].set_title('Age Distribution by Churn Status', fontweight='bold')
axes[1].set_xlabel('Age')
axes[1].set_ylabel('Density')
axes[1].legend()

plt.tight_layout()
plt.savefig('plot_tenure_age.png', bbox_inches='tight')
plt.show()

### 4.3 Usage Behavior by Churn

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, col, label in zip(axes,
                           ['calls_made', 'sms_sent', 'data_used'],
                           ['Calls Made', 'SMS Sent', 'Data Used (MB)']):
    sns.violinplot(x='churn', y=col, data=df, ax=ax,
                   palette=CHURN_COLORS, inner='quartile', cut=0)
    ax.set_xticklabels(['No Churn','Churned'])
    ax.set_title(f'{label} vs Churn', fontweight='bold')
    ax.set_xlabel('')

plt.suptitle('Usage Behavior Comparison: Churned vs Retained Customers',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plot_usage_behavior.png', bbox_inches='tight')
plt.show()

### 4.4 Gender & Dependents

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Churn rate by gender
gender_churn = df.groupby('gender')['churn'].mean() * 100
gender_churn.plot(kind='bar', ax=axes[0], color=['#E91E63','#2196F3'],
                  edgecolor='white', rot=0)
axes[0].set_title('Churn Rate by Gender', fontweight='bold')
axes[0].set_ylabel('Churn Rate (%)')
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())

# Churn rate by num_dependents
dep_churn = df.groupby('num_dependents')['churn'].mean() * 100
dep_churn.plot(kind='bar', ax=axes[1], color='#FF9800', edgecolor='white', rot=0)
axes[1].set_title('Churn Rate by Number of Dependents', fontweight='bold')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())

plt.tight_layout()
plt.savefig('plot_demographics.png', bbox_inches='tight')
plt.show()

### 4.5 Correlation Heatmap

In [ ]:
num_cols = ['age', 'num_dependents', 'estimated_salary',
            'calls_made', 'sms_sent', 'data_used',
            'tenure_months', 'usage_score', 'churn']
corr = df[num_cols].corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, linewidths=0.5,
            annot_kws={'size': 9})
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_correlation.png', bbox_inches='tight')
plt.show()
print("📌 usage_score aggregates calls + sms + data — check its churn correlation.")

### 4.6 Top States by Churn Rate

In [ ]:
top_states = (df.groupby('state')['churn']
              .mean()
              .sort_values(ascending=False)
              .head(15) * 100)

plt.figure(figsize=(12, 5))
top_states.plot(kind='bar', color='#E53935', edgecolor='white', rot=45)
plt.title('Top 15 States by Churn Rate', fontweight='bold')
plt.ylabel('Churn Rate (%)')
plt.xlabel('State')
plt.yaxis_formatter = mtick.PercentFormatter()
plt.tight_layout()
plt.savefig('plot_state_churn.png', bbox_inches='tight')
plt.show()

## 5. 🔵 Customer Segmentation via K-Means Clustering

We use behavioral + demographic features to segment customers into meaningful groups.  
Optimal k is found with the **Elbow Method**.


In [ ]:
# Features for clustering
cluster_features = ['age', 'estimated_salary', 'calls_made',
                    'sms_sent', 'data_used', 'tenure_months', 'usage_score']

X_cluster = df[cluster_features].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

# Elbow Method
inertias = []
K_range = range(2, 11)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(K_range, inertias, 'bo-', markersize=8)
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method — Optimal k for K-Means', fontweight='bold')
plt.xticks(K_range)
plt.axvline(x=4, color='red', linestyle='--', label='Chosen k=4')
plt.legend()
plt.tight_layout()
plt.savefig('plot_elbow.png', bbox_inches='tight')
plt.show()

In [ ]:
# Fit final K-Means with k=4
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_scaled)

# Cluster profiles
cluster_profile = df.groupby('cluster')[cluster_features + ['churn']].mean().round(2)
cluster_profile.index = [f'Segment {i}' for i in cluster_profile.index]
print("=== Cluster Profiles (Mean Values) ===")
cluster_profile

In [ ]:
# Visualize clusters in 2D using PCA
pca = PCA(n_components=2, random_state=42)
pca_coords = pca.fit_transform(X_scaled)

# Sample 5000 points for faster scatter plot
idx = np.random.choice(len(pca_coords), 5000, replace=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# By cluster
scatter1 = axes[0].scatter(pca_coords[idx, 0], pca_coords[idx, 1],
                            c=df['cluster'].iloc[idx], cmap='Set1',
                            alpha=0.5, s=15)
axes[0].set_title('Customer Segments (PCA View)', fontweight='bold')
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')
plt.colorbar(scatter1, ax=axes[0], label='Segment')

# By churn
churn_colors_arr = df['churn'].iloc[idx].map({0:'#2196F3', 1:'#F44336'})
axes[1].scatter(pca_coords[idx, 0], pca_coords[idx, 1],
                c=churn_colors_arr, alpha=0.4, s=15)
axes[1].set_title('Churn Overlay on PCA View', fontweight='bold')
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')
from matplotlib.patches import Patch
axes[1].legend(handles=[Patch(color='#2196F3', label='No Churn'),
                         Patch(color='#F44336', label='Churned')])

plt.tight_layout()
plt.savefig('plot_pca_clusters.png', bbox_inches='tight')
plt.show()
print(f"PCA explains {pca.explained_variance_ratio_.sum()*100:.1f}% of variance")

In [ ]:
# Churn rate per segment
seg_churn = df.groupby('cluster')['churn'].mean() * 100
seg_size  = df.groupby('cluster').size()

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar([f'Segment {i}' for i in seg_churn.index],
              seg_churn.values,
              color=['#4CAF50','#FF9800','#F44336','#2196F3'],
              edgecolor='white')
for bar, size in zip(bars, seg_size.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'n={size:,}', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('Churn Rate (%)')
ax.set_title('Churn Rate by Customer Segment', fontweight='bold')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
plt.tight_layout()
plt.savefig('plot_segment_churn.png', bbox_inches='tight')
plt.show()

## 6. ⚙️ Preprocessing for Machine Learning

In [ ]:
# Encode categoricals
le = LabelEncoder()
df_ml = df.copy()
for col in ['gender', 'telecom_partner']:
    df_ml[col] = le.fit_transform(df_ml[col])

# Drop non-predictive columns
drop_cols = ['customer_id', 'state', 'city', 'pincode',
             'date_of_registration', 'age_group']
df_ml = df_ml.drop(columns=drop_cols)

feature_cols = [c for c in df_ml.columns if c != 'churn']
X = df_ml[feature_cols]
y = df_ml['churn']

print(f"Features: {feature_cols}")
print(f"X shape: {X.shape} | y shape: {y.shape}")
print(f"Class balance: {y.value_counts().to_dict()}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train: {X_train.shape} | Test: {X_test.shape}")

## 7. 🤖 Churn Prediction Models

We run **three progressive approaches**:
1. Baseline Random Forest
2. Class-weighted RF (handle imbalance without resampling)
3. RF + SMOTE oversampling (best for recall on minority class)


### 7.1 Baseline Random Forest

In [ ]:
rf_base = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_base.fit(X_train, y_train)
pred_base = rf_base.predict(X_test)

print("=== Baseline RF ===")
print(f"Accuracy: {accuracy_score(y_test, pred_base):.4f}")
print(classification_report(y_test, pred_base, target_names=['No Churn','Churned']))

### 7.2 Class-Weighted RF (handle imbalance)

In [ ]:
rf_balanced = RandomForestClassifier(n_estimators=200, max_depth=12,
                                     class_weight='balanced',
                                     random_state=42, n_jobs=-1)
rf_balanced.fit(X_train, y_train)
pred_bal = rf_balanced.predict(X_test)

print("=== Class-Weighted RF ===")
print(f"Accuracy: {accuracy_score(y_test, pred_bal):.4f}")
print(classification_report(y_test, pred_bal, target_names=['No Churn','Churned']))

### 7.3 RF + SMOTE (Synthetic Minority Oversampling)

In [ ]:
# SMOTE on training set only — NEVER apply to test set
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f"Before SMOTE: {dict(zip(*np.unique(y_train, return_counts=True)))}")
print(f"After  SMOTE: {dict(zip(*np.unique(y_train_sm, return_counts=True)))}")

rf_smote = RandomForestClassifier(n_estimators=200, max_depth=12,
                                   random_state=42, n_jobs=-1)
rf_smote.fit(X_train_sm, y_train_sm)
pred_smote = rf_smote.predict(X_test)

print("\n=== RF + SMOTE ===")
print(f"Accuracy: {accuracy_score(y_test, pred_smote):.4f}")
print(classification_report(y_test, pred_smote, target_names=['No Churn','Churned']))

## 8. 📈 Model Evaluation

### 8.1 Confusion Matrices — All Three Models

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, pred, title in zip(axes,
                            [pred_base, pred_bal, pred_smote],
                            ['Baseline RF', 'Class-Weighted RF', 'RF + SMOTE']):
    ConfusionMatrixDisplay.from_predictions(
        y_test, pred,
        display_labels=['No Churn', 'Churned'],
        cmap='Blues', ax=ax, colorbar=False)
    ax.set_title(title, fontweight='bold')

plt.tight_layout()
plt.savefig('plot_confusion_matrices.png', bbox_inches='tight')
plt.show()

### 8.2 ROC-AUC Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for model, name, color in [
    (rf_base,     'Baseline RF',        '#9C27B0'),
    (rf_balanced, 'Class-Weighted RF',  '#2196F3'),
    (rf_smote,    'RF + SMOTE',         '#F44336')]:
    
    y_prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    ax.plot(fpr, tpr, lw=2, color=color, label=f'{name} (AUC={auc:.3f})')

ax.plot([0,1],[0,1], 'k--', lw=1, label='Random Classifier')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve Comparison', fontweight='bold')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('plot_roc_auc.png', bbox_inches='tight')
plt.show()

### 8.3 Cross-Validation (Stratified 5-Fold)

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("=== Cross-Validation Results (RF + SMOTE best model) ===\n")
for metric in ['accuracy', 'recall', 'f1', 'roc_auc']:
    scores = cross_val_score(rf_smote, X, y, cv=skf, scoring=metric, n_jobs=-1)
    print(f"{metric.upper():10s}: mean={scores.mean():.4f}  std={scores.std():.4f}  scores={np.round(scores,4)}")

### 8.4 Feature Importance

In [ ]:
importances = pd.Series(rf_smote.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(10, 5))
colors = ['#F44336' if i < 5 else '#90CAF9' for i in range(len(importances))]
importances.plot(kind='bar', color=colors, edgecolor='white')
plt.title('Feature Importance — RF + SMOTE Model', fontweight='bold')
plt.ylabel('Importance Score')
plt.xlabel('Feature')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('plot_feature_importance.png', bbox_inches='tight')
plt.show()

print("\nTop 5 Features driving churn prediction:")
for feat, score in importances.head(5).items():
    print(f"  {feat:25s} → {score:.4f}")

## 9. 📋 Model Comparison Summary

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

results = []
for name, pred, model in [
    ('Baseline RF',        pred_base,  rf_base),
    ('Class-Weighted RF',  pred_bal,   rf_balanced),
    ('RF + SMOTE',         pred_smote, rf_smote)]:
    
    y_prob = model.predict_proba(X_test)[:, 1]
    results.append({
        'Model':     name,
        'Accuracy':  round(accuracy_score(y_test, pred), 4),
        'Precision': round(precision_score(y_test, pred), 4),
        'Recall':    round(recall_score(y_test, pred), 4),
        'F1-Score':  round(f1_score(y_test, pred), 4),
        'ROC-AUC':   round(roc_auc_score(y_test, y_prob), 4)
    })

results_df = pd.DataFrame(results).set_index('Model')
results_df.style.highlight_max(axis=0, color='#C8E6C9').highlight_min(axis=0, color='#FFCDD2')

## 10. 💡 Business Insights & Recommendations

| Insight | Finding | Action |
|---|---|---|
| **High-risk segment** | Customers with low usage_score churn more | Targeted re-engagement offers |
| **Tenure matters** | Newer customers churn at higher rates | Onboarding loyalty programs in first 6 months |
| **Data heavy users stay** | High data_used → lower churn | Promote data-heavy bundles |
| **Dependents reduce churn** | Customers with 3+ dependents are stickier | Family plan incentives |
| **Geographic hotspots** | Certain states have >25% churn | Region-specific retention teams |
| **Operator parity** | Churn rates roughly equal across 4 operators | Focus on usage behavior, not operator |

### Next Steps
- Deploy best model (RF + SMOTE) as a churn scoring API
- Build a **30-day churn probability dashboard** for customer success teams
- Combine cluster labels with churn scores for hyper-personalized outreach
